# DA6401 — Kaggle Account 1: Dropout 0.0 + Dropout 0.2

**Before running:**
- Accelerator → **P100**
- Internet → **ON**
- Then: Run All → close browser → come back later

W&B tag: `kaggle-sys1`

Estimated time: ~5 hours on P100

In [ ]:
import os
os.environ["WANDB_API_KEY"]      = "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"
os.environ["WANDB_DISABLE_CODE"] = "1"

CKPT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

!git clone https://github.com/usnaveen/A2_Deep_Learning.git da6401_assignment_2 2>&1 | tail -2
%cd da6401_assignment_2
!pip install -q wandb albumentations scikit-learn

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — enable GPU!'}")

In [ ]:
from data.pets_dataset import download_oxford_pet
download_oxford_pet("./data/oxford_pet")
print("Dataset ready.")

In [ ]:
# ── Run 1: dropout = 0.0  (W&B tag: kaggle-sys1) ──
!python train.py \
    --task classify \
    --experiment kaggle-sys1 \
    --dropout 0.0 \
    --epochs 120 \
    --lr 1e-3 \
    --batch-size 32 \
    --num-workers 2 \
    --checkpoint-dir /kaggle/working/checkpoints

In [ ]:
# ── Run 2: dropout = 0.2  (W&B tag: kaggle-sys1) ──
!python train.py \
    --task classify \
    --experiment kaggle-sys1 \
    --dropout 0.2 \
    --epochs 120 \
    --lr 1e-3 \
    --batch-size 32 \
    --num-workers 2 \
    --checkpoint-dir /kaggle/working/checkpoints

In [ ]:
# ── Final check: best checkpoint F1 ──
import torch
from sklearn.metrics import f1_score
from data.pets_dataset import create_dataloaders
from models.classification import VGG11Classifier

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_, val_loader, _ = create_dataloaders(root="./data/oxford_pet", batch_size=64, num_workers=2)

# Try dropout=0.2 first, fall back to 0.0
for drop in [0.2, 0.0]:
    try:
        model = VGG11Classifier(num_classes=37, dropout_p=drop, use_bn=True).to(device)
        model.load_state_dict(torch.load("/kaggle/working/checkpoints/classifier.pth", map_location=device, weights_only=False))
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                logits = model(batch["image"].to(device))
                all_preds.extend(logits.argmax(1).cpu().tolist())
                all_labels.extend(batch["label"].tolist())
        f1 = f1_score(all_labels, all_preds, average="macro")
        print(f"\n✅ Best checkpoint Val Macro-F1 = {f1:.4f}")
        print("   Checkpoint saved at: /kaggle/working/checkpoints/classifier.pth")
        print("   Download this file from Kaggle output tab!")
        break
    except Exception as e:
        print(f"drop={drop} failed: {e}")